# Notebook 10 · Figure of qualitative examples (reference boxes and the boxes of each arm)

**What it does:** draws, on CQ500 validation slices, the reference boxes (BHX) and the boxes of each arm.
It does not train or score anything: it only reads the predictions already written by notebooks 06 and 08.

**How the examples are chosen** (rule fixed before any image is looked at):
- 4 examinations with haemorrhage drawn at random with seed 2026; in each, the slice with the largest area
  of reference boxes;
- 2 normal examinations drawn with the same generator; in each, the slice with the highest-scoring
  detection of any of the three arms.

Each arm shows the boxes above its operating point of one false positive per examination, computed on its
own validation fold (the same as in the per-subtype analysis).

**Before running:** `src/ichdet` version 0.4.0 on the Drive. No GPU is needed (*Runtime → Change runtime
type → CPU*). **Time:** about 2 minutes.

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Drive and code
import json, os, sys, zipfile, io
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['EXPERIMENTO_LOCAL'])

sys.path.insert(0, str(RAIZ / 'codigo'))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
PRIMEIRA = RAIZ / 'dados' / 'tc' / 'experimento' / 'completo'
CORRECAO = RAIZ / 'dados' / 'tc' / 'experimento' / 'correcao_retinanet'
FIGURAS = CORRECAO / 'figuras'
FIGURAS.mkdir(exist_ok=True)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import ichdet
from ichdet import data_io, experiment, final
from ichdet.metrics import COCO_IOU_THRESHOLDS, DetectionTable

print('ichdet', ichdet.__version__, '· código congelado', experiment.frozen_fingerprint(RAIZ / 'codigo' / 'ichdet'))

In [ ]:
# 2. Reference boxes of the five folds (read from inside the archive) and predictions of the three arms
A, B, C = 'retinanet', 'yolo_no_mosaic', 'yolo_mosaic'
NOME = {A: 'RetinaNet', B: 'YOLOv8m, no mosaic', C: 'YOLOv8m, mosaic'}
PASTA = {A: CORRECAO, B: PRIMEIRA, C: PRIMEIRA}
K = experiment.FOLDS

ZIP = zipfile.ZipFile(PREPARO / 'cq500_512.zip')
REF, ARQUIVO = {}, {}
for k in K:
    entradas = json.loads(ZIP.read(f'monai/fold{k}.json'))['validation']
    REF[k] = {data_io.image_key(e['image']): (e['exam'], np.asarray(e['box'], dtype=float).reshape(-1, 4)) for e in entradas}
    ARQUIVO.update({data_io.image_key(e['image']): e['image'] for e in entradas})
with open('/content/boxes_512.csv' if 'google.colab' in sys.modules else RAIZ / 'boxes_512.csv', 'wb') as f:
    f.write(ZIP.read('boxes_512.csv'))
    CAIXAS_CSV = f.name

PRED = {(k, b): data_io.load_predictions(experiment.job_dir(PASTA[b], k, b) / 'pred_validation.csv') for k in K for b in (A, B, C)}
LIMIAR = {(k, b): final.score_at_fp_rate(DetectionTable.build(REF[k], PRED[(k, b)], COCO_IOU_THRESHOLDS), 1.0)
          for k in K for b in (A, B, C)}
for k in K:
    print(f'rodada {k}: limiares a 1 FP/exame', {NOME[b]: round(LIMIAR[(k, b)], 3) for b in (A, B, C)})

In [ ]:
# 3. Random draw of the examinations and choice of the slices (rule fixed before looking at the images)
RODADA, CORTES = {}, {}
for k in K:
    for img, (exame, caixas) in REF[k].items():
        RODADA[exame] = k
        CORTES.setdefault(exame, []).append(img)
com_sangue = sorted(e for e, imgs in CORTES.items() if any(len(REF[RODADA[e]][i][1]) for i in imgs))
normais = sorted(e for e in CORTES if e not in com_sangue)
assert len(com_sangue) == 215 or 'EXPERIMENTO_LOCAL' in os.environ, len(com_sangue)

sorteio = np.random.default_rng(2026)
escolhidos_sangue = list(sorteio.choice(com_sangue, 4, replace=False))
escolhidos_normais = list(sorteio.choice(normais, 2, replace=False))

def area(caixas):
    return float(np.sum((caixas[:, 2] - caixas[:, 0]) * (caixas[:, 3] - caixas[:, 1]))) if len(caixas) else 0.0

def maior_nota(img, k):
    return max((float(PRED[(k, b)][img][1].max()) for b in (A, B, C) if img in PRED[(k, b)] and len(PRED[(k, b)][img][1])),
               default=-1.0)

exemplos = []
for exame in escolhidos_sangue:
    k = RODADA[exame]
    img = max(sorted(CORTES[exame]), key=lambda i: area(REF[k][i][1]))
    exemplos.append({'exam': exame, 'fold': k, 'image': img, 'group': 'haemorrhage', 'rule': 'largest reference box area'})
for exame in escolhidos_normais:
    k = RODADA[exame]
    img = max(sorted(CORTES[exame]), key=lambda i: maior_nota(i, k))
    exemplos.append({'exam': exame, 'fold': k, 'image': img, 'group': 'normal', 'rule': 'highest-scoring detection of any arm'})

tipos = final.box_subtypes({e['image']: REF[e['fold']][e['image']] for e in exemplos}, CAIXAS_CSV)
for e in exemplos:
    k, img = e['fold'], e['image']
    e['reference_boxes'] = len(REF[k][img][1])
    e['subtypes'] = sorted(set(tipos[img]))
    e['boxes_shown'] = {}
    for b in (A, B, C):
        caixas, notas = PRED[(k, b)].get(img, (np.zeros((0, 4)), np.zeros(0)))
        manter = notas >= LIMIAR[(k, b)]
        e['boxes_shown'][b] = int(manter.sum())
        e[f'_{b}'] = (caixas[manter], notas[manter])
    print(e['exam'], f"rodada {k}", img, e['group'], e['subtypes'], e['boxes_shown'])

In [ ]:
# 4. Figure: brain window (channel 1 of the model input), reference boxes and the boxes of each arm
COR = {A: '#2a78d6', B: '#eb6834', C: '#1baf7a'}
colunas = [('Reference (BHX)', None), (NOME[A], A), (NOME[B], B), (NOME[C], C)]
fig, eixos = plt.subplots(len(exemplos), 4, figsize=(7.0, 1.85 * len(exemplos)))
for n, (linha, e) in enumerate(zip(eixos, exemplos)):
    imagem = np.asarray(Image.open(io.BytesIO(ZIP.read(ARQUIVO[e['image']]))))[:, :, 0]
    referencia = REF[e['fold']][e['image']][1]
    for ax, (titulo, b) in zip(linha, colunas):
        ax.imshow(imagem, cmap='gray', vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        for x1, y1, x2, y2 in referencia:
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='white',
                                       lw=1.4 if b is None else 0.8, ls='-' if b is None else (0, (2, 2))))
        if b is not None:
            for x1, y1, x2, y2 in e[f'_{b}'][0]:
                ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=COR[b], lw=1.4))
        if n == 0:
            ax.set_title(titulo, fontsize=8)
    rotulo = 'Normal' if e['group'] == 'normal' else ', '.join(e['subtypes'])
    linha[0].set_ylabel(f"{e['exam']}\n{rotulo}", fontsize=7)
plt.subplots_adjust(wspace=0.03, hspace=0.06)
fig.savefig(FIGURAS / 'qualitative_examples.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIGURAS / 'qualitative_examples.png', bbox_inches='tight', dpi=150)
selecao = {'seed': 2026, 'operating_point': '1 false positive per examination, per fold model',
           'thresholds': {f'fold{k}/{b}': LIMIAR[(k, b)] for k in K for b in (A, B, C)},
           'examples': [{k: v for k, v in e.items() if not k.startswith('_')} for e in exemplos]}
(FIGURAS / 'qualitative_examples.json').write_text(json.dumps(selecao, indent=2, default=float))
print('figura gravada em', FIGURAS)

## After running

`qualitative_examples.pdf` and `qualitative_examples.json` are written to
`dados/tc/experimento/correcao_retinanet/figuras/` on the Drive; the JSON (selection and thresholds) is
included in this repository under `data/`.